## 1. Load the datasets

In [9]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification , DataCollatorWithPadding


In [10]:
raw_datasets = load_dataset("glue","mrpc")
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)


def tokenize_function(example):
  return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function , batched = True)

data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

# 2. Prepare for training

At this stage, tokenized_datasets is your tokenized Hugging Face DatasetDict, but it still contains some columns the model does not expect (e.g., the original text fields). Also, the label column name is not yet aligned with what Transformers models expect.

So we do three cleanup steps:

* 1. Remove columns the model will not use

* 2. Rename label → labels

* 3. Make the dataset return PyTorch tensors instead of Python lists

### 1. Remove unused columns

In [11]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [12]:
tokenized_datasets = tokenized_datasets.remove_columns(['sentence1', 'sentence2','idx'])

In [13]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

### Why is this needed?

* sentence1 and sentence2 are raw text; the model does not take raw strings as input. It expects tokenized features like input_ids.

* idx is just an identifier/index; it is not used for training.

* If you keep these extra columns, the DataLoader may include them in the batch, and model(**batch) can fail due to unexpected keyword arguments or mismatched inputs.

## 2) Rename label to labels

In [15]:
tokenized_datasets = tokenized_datasets.rename_column("label","labels")

In [16]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

### Why is this needed?
Transformers models typically expect this forward signature:

* input_ids=...

* attention_mask=...

* token_type_ids=... (for some models like BERT)

* labels=... ✅

If the column is still named label, then:

* the model may not receive labels correctly

* the model may not automatically compute loss

* you may have to compute loss manually

### 3) Set the dataset format to PyTorch

In [18]:
tokenized_datasets.set_format("torch")

In [19]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [20]:
tokenized_datasets['train'].shape

(3668, 4)

In [23]:
tokenized_datasets['train'].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [24]:
tokenized_datasets['train'][0]

{'labels': tensor(1),
 'input_ids': tensor([  101,  2572,  3217,  5831,  5496,  2010,  2567,  1010,  3183,  2002,
          2170,  1000,  1996,  7409,  1000,  1010,  1997,  9969,  4487, 23809,
          3436,  2010,  3350,  1012,   102,  7727,  2000,  2032,  2004,  2069,
          1000,  1996,  7409,  1000,  1010,  2572,  3217,  5831,  5496,  2010,
          2567,  1997,  9969,  4487, 23809,  3436,  2010,  3350,  1012,   102]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1])}

### What these columns mean

* input_ids: token IDs produced by the tokenizer

* attention_mask: 1 for real tokens, 0 for padding tokens

* token_type_ids: distinguishes sentence A vs sentence B (for BERT-style models)

* labels: ground-truth class (0/1)

# 3 Why do we need a DataLoader?

tokenized_datasets["train"] is a dataset containing many examples.
During training, you cannot feed the entire dataset into the model at once, so you split the data into smaller mini-batches.

A PyTorch DataLoader helps you by:

* creating batches from the dataset
* shuffling the training data each epoch (if enabled)
* handling batching + padding correctly via collate_fn
* making iteration simple: for batch in train_dataloader: ...

In [25]:
from torch.utils.data import DataLoader

### Train DataLoader

In [27]:
train_dataloader = DataLoader(
    tokenized_datasets["train"],
    shuffle = True,
    batch_size = 8,
    collate_fn = data_collator
)

Explanation of each argument

* tokenized_datasets["train"]
→ the training split dataset

* shuffle=True
→ randomizes the order of examples each epoch
Why? If the order is always the same, the model can learn order-specific patterns and generalize worse.

* batch_size=8
→ each training step processes 8 examples at once
(chosen based on GPU/CPU memory constraints)

* collate_fn=data_collator (very important)
→ builds a proper batch from individual examples
Why needed? Sequence lengths differ across examples.
The collator:

  * finds the maximum sequence length in the batch
  * pads shorter sequences to match that max length
  * stacks everything into tensors

### Evaluation DataLoader

In [28]:
eval_loader = DataLoader(
    tokenized_datasets["validation"],
    batch_size = 8,
    collate_fn = data_collator
)

In [29]:
eval_loader

Here, shuffle=False by default.

**Why not shuffle for evaluation?**

* evaluation does not need random order
* keeping order stable improves reproducibility and consistency of results

### 3) Quick sanity check: does batching work?

In [31]:
for batch in train_dataloader:
  break

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 81]),
 'token_type_ids': torch.Size([8, 81]),
 'attention_mask': torch.Size([8, 81])}

This does not train the model. It only:

* fetches the first batch from the DataLoader
* stops immediately with break

So the purpose is: “show me one batch”.

### 4) Inspect tensor shapes inside the batch

In [32]:
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([8]),
 'input_ids': torch.Size([8, 81]),
 'token_type_ids': torch.Size([8, 81]),
 'attention_mask': torch.Size([8, 81])}

### What these shapes mean

**Why** [8, 65]?

* 8 = batch size (8 examples)

* 81 = the maximum token length within that batch (after padding)

So:

* input_ids: 8 examples padded to length 81

* attention_mask: same shape; indicates real tokens vs padding

* token_type_ids: same shape for BERT-style models

**Why** are labels [8]?

* one label per example

* 8 examples → 8 labels

# 4. Model initialization + sanity forward pass + optimizer

In [34]:
from transformers import AutoModelForSequenceClassification

checkpoint = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


** What’s happening?**

* AutoModelForSequenceClassification is a wrapper that:

  * inspects the checkpoint (BERT / RoBERTa / etc.)
  * loads the correct base Transformer architecture
  * adds a sequence classification head on top

* .from_pretrained(checkpoint, ...) loads the pretrained weights from that checkpoint.

* num_labels=2 means this is binary classification (two classes, e.g., 0/1).

### 2) Pass one batch through the model (sanity check)

In [35]:
outputs = model(**batch)

In [36]:
print(outputs.loss,outputs.logits.shape)

tensor(0.7721, grad_fn=<NllLossBackward0>) torch.Size([8, 2])


** What does model(**batch) mean?**

* batch is a dictionary, typically containing:

  * batch["input_ids"]
  *batch["attention_mask"]
  *batch["token_type_ids"]
  *batch["labels"]

* Using **batch unpacks the dictionary into keyword arguments, roughly equivalent to:

In [37]:
outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    token_type_ids=batch["token_type_ids"],
    labels=batch["labels"]
)

**Why is this important?**
This verifies that:

* your dataset columns match what the model expects
* the DataLoader is producing valid batches
* the model forward pass works
* the model can compute loss correctly (meaning labels are wired properly)

### **3) Why do we need an optimizer?**

In a training loop, after you compute the loss:

* loss.backward() computes gradients
* the optimizer uses those gradients to update the model weights

Without an optimizer, the model will never actually learn/update.

# 4) Set up the AdamW optimizer

In [39]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(),lr = 5e-5)

**What’s happening?**

* model.parameters() provides all trainable weights.

* AdamW is the standard optimizer for Transformers fine-tuning:

  * similar to Adam
  * but handles weight decay in a better (decoupled) way

* lr=5e-5 is a common default learning rate (also used by Trainer by default).

**Learning rate meaning**

* controls how large each weight update step is:

  * too high → unstable training
  * too low → slow learning / underfitting

# 5 Why do we need a scheduler?

### **1) Why a learning rate scheduler is useful**

The optimizer (AdamW) updates model weights, but if you keep the learning rate constant, training is often less effective because:

* early in training you may want larger updates (learn quickly)

* later in training you usually want smaller updates (fine-tune carefully)

That’s why the Trainer commonly uses a learning rate scheduler by default.

**Default behavior used here: Linear Decay**

* starts at lr = 5e-5

* decreases linearly during training

* ends near 0 at the final step


### **2) Why do we need num_training_steps?**

A scheduler must know how many total update steps will happen so it can plan the full learning rate curve.

* One training step = processing one batch + calling optimizer.step()

* One epoch contains len(train_dataloader) batches

So total steps:

num_training_steps=
num_epochs
×
len(train_dataloader)


In [40]:
from transformers import get_scheduler

num_epochs = 3 # default 3 epochs
num_traning_steps = num_epochs * len(train_dataloader)

In [41]:
num_traning_steps

1377

Meaning:

* across the full training run, the optimizer will update weights 1377 times

### **4) Create the scheduler**

In [44]:
lr_scheduler = get_scheduler(
    "linear",
    optimizer = optimizer,
    num_warmup_steps = 0,
    num_training_steps = num_traning_steps

)

What each argument means

* "linear": learning rate decreases linearly
5e-5 → 0

* optimizer=optimizer: which optimizer’s learning rate to control

* num_warmup_steps=0: no warmup phase (warmup would increase lr gradually at the beginning)

* num_training_steps=num_training_steps: total number of steps over which lr will decay to (near) zero

# 6.Training loop core (device + progress bar + backprop)

### 1) Set up the GPU/CPU device

In [48]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [49]:
device

device(type='cpu')

### What’s happening?

* torch.cuda.is_available() checks whether a CUDA GPU is available.

* If yes → device = "cuda", otherwise → device = "cpu".

* model.to(device) moves all model parameters/weights to that device.

**Why is this necessary?**

Model and data must be on the same device. If the model is on GPU but the batch is on CPU (or vice versa), you’ll get an error like:

Expected all tensors to be on the same device...

### 2) Create a progress bar with tqdm


In [51]:
from tqdm.auto import tqdm

progress_bar = tqdm(range(num_traning_steps))

  0%|          | 0/1377 [00:00<?, ?it/s]

**Why do this?**

* To visually track training progress.

* num_training_steps is the total number of optimizer update steps (epochs × number of train batches).

* tqdm(range(...)) creates a progress bar that you will manually update each training step.

In [55]:
progress_bar

### 3) Switch the model to training mode

In [56]:
model.train()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

**What does this do?**
It sets the model to training mode, which affects certain layers:

  * Dropout: ON during training, OFF during evaluation
  * BatchNorm (if present): uses training behavior/statistics

So you should call model.train() before starting the training loop.

### 4) Training loop structure: epochs + batches

In [ ]:
for epoch in range(num_epochs):
    for batch in train_dataloader:
        ...

Concept

* You iterate over the full dataset num_epochs times.

* In each epoch, the DataLoader yields batches one by one.

### 5) Move each batch to the same device as the model

In [57]:
batch = {k: v.to(device) for k, v in batch.items()}

In [58]:
batch

{'labels': tensor([1, 0, 1, 0, 0, 1, 0, 1]),
 'input_ids': tensor([[  101,  1996,  4034,  2056,  2009,  2052,  2025,  2713,  1996,  2171,
           1997,  1996,  2459,  1011,  2095,  1011,  2214,  2920,  2138,  2002,
           2003,  1037,  3576,  1012,   102,  1996,  4034,  2056,  2009,  2018,
           2053,  3488,  2000,  2713,  1996,  2171,  1997,  1996, 10563,  2920,
           2138,  2002,  2001,  1037,  3576,  1012,   102,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0],
         [  101,  1037,  3026,  2250,  2486,  2880,  1998,  1037,  2266,  1997,
           3323,  2020,  2036,  2006,  2604,  1010,  2110,  2557,  3189,  2056,
           1012,   102,  1037,  3026,  2250,  2486,  2880,  1998,  1037,  2266,
           1997,  3323,  2036,  2351,  1999,

**What’s happening?**

* batch is a dictionary containing tensors like:

  * input_ids
  * attention_mask
  * token_type_ids
  * labels

Each tensor is moved to GPU/CPU using .to(device).

**Why needed?**
Because the model is on device, and inputs must match the device.

### 6) Forward pass (prediction + loss)


In [61]:
output = model(**batch)
loss = output.loss

In [62]:
loss

tensor(0.6725, grad_fn=<NllLossBackward0>)

### 7) Backpropagation (compute gradients)


In [63]:
loss.backward()

In [64]:
loss

tensor(0.6725, grad_fn=<NllLossBackward0>)



**Important**

* At this stage, weights are not updated yet.
* You only have gradients computed.

### 8) Update weights + update scheduler + reset gradients

In [65]:
optimizer.step()
lr_scheduler.step()
optimizer.zero_grad()

optimizer.step()
* Uses gradients to update weights (AdamW update rule).

lr_scheduler.step()
* Updates the learning rate according to the schedule (linear decay: 5e-5 → 0).

Common practice is:
* optimizer.step() then lr_scheduler.step()

optimizer.zero_grad()
* Clears gradients from the previous step.

**Why clear gradients?**
PyTorch accumulates gradients by default.
If you don’t clear them, gradients from the next batch will add on top of the previous batch gradients (usually unintended).

### 9) Update the progress bar

In [66]:
progress_bar.update(1)

True

### 10) Why this loop has no evaluation/reporting

This training loop only:

* computes loss
* backpropagates
* updates weights

It does not measure model quality (accuracy/F1).